<a href="https://colab.research.google.com/github/vigneshhh34/Heritage-Lens/blob/main/notebooks/03_original_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT = '/content/drive/MyDrive/heritage-lens'
print(os.listdir(PROJECT))

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
print('Kaggle token loaded')

In [ ]:
for ds in ["jungseolin/international-architectural-styles-combined",
           "gustavoachavez/architectural-styles-periods-dataset"]:
    print("=====", ds)
    !kaggle datasets files -d {ds} | head -n 15

In [ ]:
import os, glob
import matplotlib.pyplot as plt

ROOT = '/content/data'

# README
print(open(f'{ROOT}/README.txt').read()[:1500])
print('-' * 60)

# count images per style
classes = sorted(d for d in os.listdir(ROOT) if os.path.isdir(f'{ROOT}/{d}'))
counts = {}
for c in classes:
    files = glob.glob(f'{ROOT}/{c}/data/**/*.*', recursive=True)
    counts[c] = len([f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

print(len(classes), 'styles,', sum(counts.values()), 'images in total')
for c, n in sorted(counts.items(), key=lambda x: x[1]):
    print(f'{n:6d}  {c}')

# chart
items = sorted(counts.items(), key=lambda x: x[1])
plt.figure(figsize=(9, max(5, len(items) * 0.28)))
plt.barh([k.replace(' architecture', '') for k, v in items], [v for k, v in items])
plt.xlabel('Number of images')
plt.title('Images per architectural style')
plt.tight_layout()
plt.show()

In [ ]:
!kaggle datasets download -d jungseolin/international-architectural-styles-combined -p /content/data --unzip
!find /content/data -maxdepth 2 | head -n 40

In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

ROOT = '/content/data'
PROJECT = '/content/drive/MyDrive/heritage-lens'
MIN_IMAGES = 150

# 1. list every image with its style
rows = []
for style in sorted(d for d in os.listdir(ROOT) if os.path.isdir(f'{ROOT}/{d}')):
    for p in glob.glob(f'{ROOT}/{style}/data/**/*.*', recursive=True):
        if p.lower().endswith(('.jpg', '.jpeg', '.png')):
            rows.append({'path': os.path.relpath(p, ROOT), 'label': style})
df = pd.DataFrame(rows)
print('Found:', len(df), 'images,', df.label.nunique(), 'styles')

# 2. drop tiny classes
counts = df.label.value_counts()
dropped = counts[counts < MIN_IMAGES]
print('Dropping:', dict(dropped))
df = df[df.label.isin(counts[counts >= MIN_IMAGES].index)].copy()

# 3. remove unreadable images
def readable(rel):
    try:
        with Image.open(f'{ROOT}/{rel}') as im:
            im.verify()
        return True
    except Exception:
        return False
df['ok'] = df.path.map(readable)
print('Corrupt images removed:', (~df.ok).sum())
df = df[df.ok].drop(columns='ok').reset_index(drop=True)

# 4. stratified 80/10/10 split
train, temp = train_test_split(df, test_size=0.2, stratify=df.label, random_state=42)
val, test = train_test_split(temp, test_size=0.5, stratify=temp.label, random_state=42)
train['split'], val['split'], test['split'] = 'train', 'val', 'test'
final = pd.concat([train, val, test]).reset_index(drop=True)

os.makedirs(f'{PROJECT}/datasets', exist_ok=True)
final.to_csv(f'{PROJECT}/datasets/splits.csv', index=False)

print('\nStyles kept:', final.label.nunique())
print(final.split.value_counts())

# 5. quick look at image sizes (sample of 300)
sizes = [Image.open(f'{ROOT}/{p}').size for p in final.sample(300, random_state=1).path]
w, h = zip(*sizes)
print(f'\nMedian image size: {int(pd.Series(w).median())} x {int(pd.Series(h).median())}')
plt.scatter(w, h, s=8); plt.xlabel('width'); plt.ylabel('height')
plt.title('Image sizes (sample of 300)'); plt.show()

In [ ]:
import os, json, time
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

ROOT = '/content/data'
PROJECT = '/content/drive/MyDrive/heritage-lens'
assert os.path.exists(ROOT), 'Data folder missing: rerun the Kaggle download cell first'
dev = 'cuda'
EPOCHS, BATCH = 8, 64

# ---------- data ----------
df = pd.read_csv(f'{PROJECT}/datasets/splits.csv')
classes = sorted(df.label.unique())
c2i = {c: i for i, c in enumerate(classes)}
os.makedirs(f'{PROJECT}/models', exist_ok=True)
json.dump(classes, open(f'{PROJECT}/models/classes.json', 'w'))

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), norm])
eval_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), norm])

class DS(Dataset):
    def __init__(self, d, tf):
        self.p, self.tf = d.path.tolist(), tf
        self.y = [c2i[l] for l in d.label]
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        img = Image.open(f'{ROOT}/{self.p[i]}').convert('RGB')
        return self.tf(img), self.y[i]

tr, va = df[df.split == 'train'], df[df.split == 'val']
train_dl = DataLoader(DS(tr, train_tf), BATCH, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
val_dl = DataLoader(DS(va, eval_tf), BATCH, shuffle=False, num_workers=4, pin_memory=True)

# class weights (rarer styles count more)
cnt = tr.label.value_counts().reindex(classes).values
weights = torch.tensor(cnt.sum() / (len(classes) * cnt), dtype=torch.float32).to(dev)

# ---------- model ----------
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, len(classes)))
model = model.to(dev)

loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda')

# ---------- resume if a checkpoint exists ----------
LAST, BEST = f'{PROJECT}/models/last.pt', f'{PROJECT}/models/best.pt'
start, best, hist = 0, 0.0, {'loss': [], 'val_acc': [], 'val_top3': []}
if os.path.exists(LAST):
    ck = torch.load(LAST, map_location=dev, weights_only=False)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
    start, best, hist = ck['epoch'], ck['best'], ck['hist']
    print(f'Resuming from epoch {start}')

def evaluate(dl):
    model.eval(); c1 = c3 = n = 0
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for x, y in dl:
            out = model(x.to(dev)); y = y.to(dev)
            top3 = out.topk(3, 1).indices
            c1 += (top3[:, 0] == y).sum().item()
            c3 += (top3 == y[:, None]).any(1).sum().item()
            n += len(y)
    return c1 / n, c3 / n

# ---------- train ----------
for ep in range(start, EPOCHS):
    model.train(); t0 = time.time(); run = 0; steps = 0
    for x, y in train_dl:
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            loss = loss_fn(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        run += loss.item(); steps += 1
    sched.step()
    acc, top3 = evaluate(val_dl)
    hist['loss'].append(run / steps); hist['val_acc'].append(acc); hist['val_top3'].append(top3)
    print(f'Epoch {ep+1}/{EPOCHS} | loss {run/steps:.3f} | val acc {acc:.3f} | val top-3 {top3:.3f} | {time.time()-t0:.0f}s')
    if acc > best:
        best = acc; torch.save({'model': model.state_dict(), 'classes': classes}, BEST)
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'epoch': ep + 1, 'best': best, 'hist': hist}, LAST)

# ---------- curves for the report ----------
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist['loss']); ax[0].set_title('Training loss'); ax[0].set_xlabel('epoch')
ax[1].plot(hist['val_acc'], label='top-1'); ax[1].plot(hist['val_top3'], label='top-3')
ax[1].set_title('Validation accuracy'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()
print('Best val accuracy:', best)

In [ ]:
import os, json, numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             f1_score, classification_report, confusion_matrix)

ROOT = '/content/data'
PROJECT = '/content/drive/MyDrive/heritage-lens'
dev = 'cuda'
assert os.path.exists(ROOT), 'Data missing: rerun the Kaggle download cell'
os.makedirs(f'{PROJECT}/outputs', exist_ok=True)

df = pd.read_csv(f'{PROJECT}/datasets/splits.csv')
classes = json.load(open(f'{PROJECT}/models/classes.json'))
c2i = {c: i for i, c in enumerate(classes)}
short = [c.replace(' architecture', '').replace('Architecture of the ', '').replace('Architecture of ', '') for c in classes]
test = df[df.split == 'test'].reset_index(drop=True)

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
eval_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), norm])

class DS(Dataset):
    def __init__(self, d, tf):
        self.p, self.tf = d.path.tolist(), tf
        self.y = [c2i[l] for l in d.label]
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        return self.tf(Image.open(f'{ROOT}/{self.p[i]}').convert('RGB')), self.y[i]

dl = DataLoader(DS(test, eval_tf), 64, shuffle=False, num_workers=2)

# load best model
model = models.resnet50()
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, len(classes)))
ck = torch.load(f'{PROJECT}/models/best.pt', map_location=dev, weights_only=False)
model.load_state_dict(ck['model']); model.to(dev).eval()

probs, labels = [], []
with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
    for x, y in dl:
        probs.append(torch.softmax(model(x.to(dev)).float(), 1).cpu()); labels += y.tolist()
probs = torch.cat(probs).numpy(); labels = np.array(labels); preds = probs.argmax(1)
top3 = np.argsort(-probs, 1)[:, :3]

# ---- headline numbers ----
p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
print(f'Test images       : {len(labels)}')
print(f'Top-1 accuracy    : {accuracy_score(labels, preds):.3f}')
print(f'Top-3 accuracy    : {np.mean([l in t for l, t in zip(labels, top3)]):.3f}')
print(f'Macro precision   : {p:.3f}')
print(f'Macro recall      : {r:.3f}')
print(f'Macro F1          : {f1:.3f}')
print(f'Weighted F1       : {f1_score(labels, preds, average="weighted"):.3f}')

# ---- per-style report ----
rep = pd.DataFrame(classification_report(labels, preds, target_names=short,
                   output_dict=True, zero_division=0)).T.iloc[:len(classes)]
rep = rep.sort_values('f1-score')
rep.round(3).to_csv(f'{PROJECT}/outputs/per_style_report.csv')
print('\n--- 8 hardest styles (lowest F1) ---'); print(rep.head(8).round(3))
print('\n--- 8 easiest styles (highest F1) ---'); print(rep.tail(8).round(3))

# ---- most confused pairs ----
cm = confusion_matrix(labels, preds, labels=range(len(classes)))
off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted([(off[i, j], short[i], short[j]) for i in range(len(classes)) for j in range(len(classes)) if off[i, j] > 0], reverse=True)[:10]
print('\n--- Most confused pairs (true -> predicted) ---')
for n, a, b in pairs: print(f'{n:3d}  {a}  ->  {b}')

# ---- confusion matrix (row-normalised) ----
cmn = cm / cm.sum(1, keepdims=True)
plt.figure(figsize=(16, 13))
sns.heatmap(cmn, cmap='Blues', xticklabels=short, yticklabels=short, vmin=0, vmax=1)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix (share of each true style)')
plt.tight_layout(); plt.savefig(f'{PROJECT}/outputs/confusion_matrix.png', dpi=150); plt.show()

# ---- some mistakes ----
wrong = np.where(preds != labels)[0]
pick = np.random.default_rng(0).choice(wrong, 12, replace=False)
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, i in zip(axes.flat, pick):
    ax.imshow(Image.open(f'{ROOT}/{test.path[i]}').convert('RGB')); ax.axis('off')
    ax.set_title(f'True: {short[labels[i]]}\nPred: {short[preds[i]]} ({probs[i].max():.0%})', fontsize=9)
plt.tight_layout(); plt.savefig(f'{PROJECT}/outputs/mistakes.png', dpi=150); plt.show()

# save raw predictions
out = test[['path', 'label']].copy()
out['pred'] = [classes[i] for i in preds]
out.to_csv(f'{PROJECT}/outputs/test_predictions.csv', index=False)

In [ ]:
import os, json, glob, numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms

PROJECT = '/content/drive/MyDrive/heritage-lens'
dev = 'cuda'
classes = json.load(open(f'{PROJECT}/models/classes.json'))
short = [c.replace(' architecture', '').replace('Architecture of the ', '').replace('Architecture of ', '') for c in classes]

model = models.resnet50()
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, len(classes)))
model.load_state_dict(torch.load(f'{PROJECT}/models/best.pt', map_location=dev, weights_only=False)['model'])
model.to(dev).eval()

tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def predict_style(img, k=3):
    if isinstance(img, str): img = Image.open(img)
    img = img.convert('RGB')
    with torch.no_grad():
        p = torch.softmax(model(tf(img).unsqueeze(0).to(dev)), 1)[0].cpu()
    top = p.topk(k)
    return [(short[i], float(v)) for v, i in zip(top.values, top.indices)]

files = sorted(f for f in glob.glob(f'{PROJECT}/test_images/*.*')
               if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')))
print(len(files), 'images found')
cols = 4; rows = max(1, (len(files) + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))
axes = np.atleast_1d(axes).flatten()
for ax in axes: ax.axis('off')
for ax, f in zip(axes, files):
    res = predict_style(f)
    ax.imshow(Image.open(f).convert('RGB'))
    ax.set_title(os.path.basename(f)[:30] + '\n' + '\n'.join(f'{n}: {p:.0%}' for n, p in res), fontsize=9)
plt.tight_layout()
plt.savefig(f'{PROJECT}/outputs/real_photo_results.png', dpi=150)
plt.show()

In [ ]:
DESC = {
 'Achaemenid': 'Ancient Persian (c. 550-330 BC): monumental stone columns, grand staircases and relief carvings, as at Persepolis.',
 'Amazigh': 'Berber architecture of North Africa: fortified earthen villages, kasbahs and mud-brick towers.',
 'American Foursquare': 'Boxy American house (c. 1895-1930): hipped roof, wide front porch, central dormer.',
 'American craftsman style': 'Early 20th-century American homes: low gabled roofs, wide eaves, exposed rafters, tapered porch columns.',
 'Ancient Egyptian': 'Massive stone temples and pyramids with huge columns, pylon gateways and hieroglyph carvings.',
 'Edo Period': 'Japanese (1603-1868): wooden buildings, tiled roofs, castles and tea houses.',
 'Joseon Dynasty': 'Korean (1392-1897): curved tiled roofs, timber frames, palaces and hanok houses.',
 'Qing Dynasty': 'Chinese (1644-1912): imperial palaces with glazed-tile roofs, red walls and ornate wooden brackets.',
 'Art Deco': '1920s-30s: geometric, streamlined forms, stepped skyscraper tops and bold decoration.',
 'Art Nouveau': 'c. 1890-1910: flowing organic curves, floral ironwork and decorative glass.',
 'Balinese': 'Indonesian temples and compounds: carved stone gateways, tiered thatched shrines, open pavilions.',
 'Baroque': '17th-18th century European: dramatic curves, rich ornament, grand domes and facades.',
 'Bauhaus': 'German functional modernism (1919-33): plain cubic forms, flat roofs, white walls, large windows.',
 'Beaux-Arts': 'Late 19th century classical grandeur: symmetrical facades, columns, sculpture, mansard roofs.',
 'Brutalist': 'Mid-20th century: raw exposed concrete in massive, blocky forms.',
 'Buddhist': 'Temples, stupas, pagodas and monasteries with symmetrical layouts and Buddha imagery.',
 'Byzantine': 'Eastern Roman: large domes on round arches, rich mosaics, as in Hagia Sophia.',
 'Chicago school': 'Late 19th century early skyscrapers: steel frames, large windows, little ornament.',
 'Colonial': 'Settler-era style: symmetrical fronts, simple classical details, columned porches.',
 'Deconstructivism': 'Late 20th century: fragmented, angular, distorted shapes that break the regular box.',
 'Dravidian': 'South Indian temples: towering carved gopuram gateways, pillared halls and temple tanks.',
 'Edwardian': 'Early 1900s British: lighter than Victorian, red brick, big windows, restrained ornament.',
 'Georgian': 'British (1714-1830): symmetrical brick fronts, sash windows, classical proportions.',
 'Gothic': 'Medieval European: pointed arches, ribbed vaults, flying buttresses and tall spires.',
 'Greek Revival': 'Early-mid 19th century: temple-like fronts with columns and triangular pediments.',
 'Hausa': 'West African mud-brick buildings with decorative relief patterns.',
 'International style': 'Mid-20th century: glass-and-steel boxes, flat roofs, no ornament.',
 'Khmer': 'Cambodian (Angkor): sandstone temple mountains, towers, galleries and carved reliefs.',
 'Maya': 'Mesoamerican: stepped stone pyramids, temple platforms and corbelled vaults.',
 'Minangkabau': 'West Sumatra: rumah gadang houses with sweeping upturned horn-shaped roofs.',
 'Moorish': 'Islamic Iberia and North Africa: horseshoe arches, geometric tilework, courtyards.',
 'Mughal': 'Indo-Islamic (16th-18th century): onion domes, white marble, minarets, symmetrical layout, as in the Taj Mahal.',
 'Novelty': 'Buildings shaped like objects or with playful, eye-catching roadside designs.',
 'Ottoman': 'Large central domes with semi-domes and slender pencil-shaped minarets.',
 'Palladian': 'Inspired by Palladio: strict symmetry, temple-front porticos, classical villas.',
 'Postmodern': 'Late 20th century: playful mix of historical references, bold colours and ornament.',
 'Queen Anne': 'Late Victorian houses: asymmetry, turrets, wraparound porches, decorative shingles.',
 'Romanesque': '11th-12th century: thick walls, round arches, small windows, sturdy towers.',
 'Russian Revival': '19th century: onion domes, bright colours and ornamental patterns from medieval Russian churches.',
 'Safavid': 'Persian (1501-1736): tiled domes, iwans, blue and turquoise mosaics, as in Isfahan.',
 'Stalinist': 'Soviet mid-century: monumental symmetrical buildings and tiered "wedding cake" towers.',
 'Swahili': 'East African coast: coral-stone buildings, carved wooden doors, Islamic influence.',
 'Tudor Revival': 'Half-timbering, steep gables, tall chimneys and decorative brickwork.',
}
missing = [s for s in short if s not in DESC]
print('Styles without a description:', missing)   # should print []

import gradio as gr

def run(img):
    if img is None:
        return {}, 'Please upload a photo.', ''
    res = predict_style(img, k=3)
    name, p = res[0]
    if p >= 0.60:
        msg = f'Looks like {name} architecture ({p:.0%} confident).'
    elif p >= 0.35:
        msg = f'Probably {name}, but not certain ({p:.0%}). Check the other guesses.'
    else:
        msg = (f'Not sure. Best guess is {name} ({p:.0%}). The photo may be old or damaged, '
               f'or the style may not be one of the 43 the model knows.')
    return {n: pr for n, pr in res}, msg, DESC.get(name, '')

demo = gr.Interface(
    fn=run,
    inputs=gr.Image(type='pil', label='Upload a photo of a building'),
    outputs=[gr.Label(num_top_classes=3, label='Top 3 styles'),
             gr.Textbox(label='Result'),
             gr.Textbox(label='About the top guess')],
    title='Heritage Lens: Architectural Style Finder',
    description='Upload a photo (old or new) of a building and the model guesses its architectural style.',
)
demo.launch(share=True, debug=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install gradio transformers

import os, json, glob, torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from transformers import CLIPModel, CLIPProcessor

PROJECT = '/content/drive/MyDrive/heritage-lens'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev)

# --- style classifier ---
classes = json.load(open(f'{PROJECT}/models/classes.json'))
short = [c.replace(' architecture', '').replace('Architecture of the ', '').replace('Architecture of ', '') for c in classes]

model = models.resnet50()
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, len(classes)))
ck = torch.load(f'{PROJECT}/models/best.pt', map_location=dev, weights_only=False)
model.load_state_dict(ck['model']); model.to(dev).eval()

tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def predict_style(img, k=3):
    if isinstance(img, str): img = Image.open(img)
    img = img.convert('RGB')
    with torch.no_grad():
        p = torch.softmax(model(tf(img).unsqueeze(0).to(dev)), 1)[0].cpu()
    top = p.topk(k)
    return [(short[i], float(v)) for v, i in zip(top.values, top.indices)]

# --- "is this a building?" check (CLIP) ---
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(dev).eval()
clip_proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

LABELS = [
    'a photo of a building',                   # first 3 count as "building"
    'a photo of a historic monument or temple',
    'a photo of a ruined or damaged building',
    'a photo of a person',
    'a selfie or portrait of a face',
    'a photo of an animal',
    'a photo of food',
    'a photo of a landscape or nature',
    'a photo of a vehicle',
    'a photo of an indoor room',
    'a screenshot or a document',
    'a photo of an everyday object',
]
N_BUILD = 3
BUILDING_MIN = 0.5

def building_score(img):
    inputs = clip_proc(text=LABELS, images=img.convert('RGB'), return_tensors='pt', padding=True).to(dev)
    with torch.no_grad():
        p = clip(**inputs).logits_per_image.softmax(1)[0].cpu()
    return float(p[:N_BUILD].sum()), LABELS[int(p.argmax())]

print('Ready')

In [ ]:
import os
# Ensure the data directory exists and has the dataset unzipped
if not os.path.exists('/content/data') or len(os.listdir('/content/data')) < 5:
    print('Downloading and unzipping dataset...')
    !kaggle datasets download -d jungseolin/international-architectural-styles-combined -p /content/data --unzip
else:
    print('Dataset already exists.')

In [ ]:
import os, json, numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             f1_score, classification_report, confusion_matrix)

ROOT = '/content/data'
PROJECT = '/content/drive/MyDrive/heritage-lens'
dev = 'cuda'
assert os.path.exists(ROOT), 'Data missing: please check if /content/data exists'
os.makedirs(f'{PROJECT}/outputs', exist_ok=True)

df = pd.read_csv(f'{PROJECT}/datasets/splits.csv')
classes = json.load(open(f'{PROJECT}/models/classes.json'))
c2i = {c: i for i, c in enumerate(classes)}
short = [c.replace(' architecture', '').replace('Architecture of the ', '').replace('Architecture of ', '') for c in classes]
test = df[df.split == 'test'].reset_index(drop=True)

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
eval_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), norm])

class DS(Dataset):
    def __init__(self, d, tf):
        self.p, self.tf = d.path.tolist(), tf
        self.y = [c2i[l] for l in d.label]
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        return self.tf(Image.open(f'{ROOT}/{self.p[i]}').convert('RGB')), self.y[i]

# Set num_workers to 0 to prevent process deadlocks or hanging in Colab
dl = DataLoader(DS(test, eval_tf), 64, shuffle=False, num_workers=0)

# load best model
model = models.resnet50()
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, len(classes)))
ck = torch.load(f'{PROJECT}/models/best.pt', map_location=dev, weights_only=False)
model.load_state_dict(ck['model']); model.to(dev).eval()

probs, labels = [], []
with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
    for x, y in dl:
        probs.append(torch.softmax(model(x.to(dev)).float(), 1).cpu()); labels += y.tolist()
probs = torch.cat(probs).numpy(); labels = np.array(labels); preds = probs.argmax(1)
top3 = np.argsort(-probs, 1)[:, :3]

# ---- headline numbers ----
p_score, r_score, f1_score_val, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
print(f'Test images       : {len(labels)}')
print(f'Top-1 accuracy    : {accuracy_score(labels, preds):.3f}')
print(f'Top-3 accuracy    : {np.mean([l in t for l, t in zip(labels, top3)]):.3f}')
print(f'Macro precision   : {p_score:.3f}')
print(f'Macro recall      : {r_score:.3f}')
print(f'Macro F1          : {f1_score_val:.3f}')
print(f'Weighted F1       : {f1_score(labels, preds, average="weighted"):.3f}')

# ---- per-style report ----
rep = pd.DataFrame(classification_report(labels, preds, target_names=short,
                   output_dict=True, zero_division=0)).T.iloc[:len(classes)]
rep = rep.sort_values('f1-score')
rep.round(3).to_csv(f'{PROJECT}/outputs/per_style_report.csv')
print("\n--- 8 hardest styles (lowest F1) ---"); print(rep.head(8).round(3))
print("\n--- 8 easiest styles (highest F1) ---"); print(rep.tail(8).round(3))

# ---- most confused pairs ----
cm = confusion_matrix(labels, preds, labels=range(len(classes)))
off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted([(off[i, j], short[i], short[j]) for i in range(len(classes)) for j in range(len(classes)) if off[i, j] > 0], reverse=True)[:10]
print("\n--- Most confused pairs (true -> predicted) ---")
for n, a, b in pairs: print(f'{n:3d}  {a}  ->  {b}')

# ---- confusion matrix (row-normalised) ----
cmn = cm / cm.sum(1, keepdims=True)
plt.figure(figsize=(16, 13))
sns.heatmap(cmn, cmap='Blues', xticklabels=short, yticklabels=short, vmin=0, vmax=1)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix (share of each true style)')
plt.tight_layout(); plt.savefig(f'{PROJECT}/outputs/confusion_matrix.png', dpi=150); plt.show()

# ---- some mistakes ----
wrong = np.where(preds != labels)[0]
pick = np.random.default_rng(0).choice(wrong, 12, replace=False)
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, i in zip(axes.flat, pick):
    ax.imshow(Image.open(f'{ROOT}/{test.path[i]}').convert('RGB')); ax.axis('off')
    ax.set_title(f'True: {short[labels[i]]}\nPred: {short[preds[i]]} ({probs[i].max():.0%})', fontsize=9)
plt.tight_layout(); plt.savefig(f'{PROJECT}/outputs/mistakes.png', dpi=150); plt.show()

# save raw predictions
out = test[['path', 'label']].copy()
out['pred'] = [classes[i] for i in preds]
out.to_csv(f'{PROJECT}/outputs/test_predictions.csv', index=False)

In [ ]:
for f in sorted(glob.glob(f'{PROJECT}/test_images/*.*')):
    s, c = building_score(Image.open(f))
    print(f'{os.path.basename(f):20s} building score {s:.2f}   closest: {c}')

In [ ]:
import gradio as gr

DESC = {
 'Achaemenid': 'Ancient Persian (c. 550-330 BC): monumental stone columns, grand staircases and relief carvings, as at Persepolis.',
 'Amazigh': 'Berber architecture of North Africa: fortified earthen villages, kasbahs and mud-brick towers.',
 'American Foursquare': 'Boxy American house (c. 1895-1930): hipped roof, wide front porch, central dormer.',
 'American craftsman style': 'Early 20th-century American homes: low gabled roofs, wide eaves, exposed rafters, tapered porch columns.',
 'Ancient Egyptian': 'Massive stone temples and pyramids with huge columns, pylon gateways and hieroglyph carvings.',
 'Edo Period': 'Japanese (1603-1868): wooden buildings, tiled roofs, castles and tea houses.',
 'Joseon Dynasty': 'Korean (1392-1897): curved tiled roofs, timber frames, palaces and hanok houses.',
 'Qing Dynasty': 'Chinese (1644-1912): imperial palaces with glazed-tile roofs, red walls and ornate wooden brackets.',
 'Art Deco': '1920s-30s: geometric, streamlined forms, stepped skyscraper tops and bold decoration.',
 'Art Nouveau': 'c. 1890-1910: flowing organic curves, floral ironwork and decorative glass.',
 'Balinese': 'Indonesian temples and compounds: carved stone gateways, tiered thatched shrines, open pavilions.',
 'Baroque': '17th-18th century European: dramatic curves, rich ornament, grand domes and facades.',
 'Bauhaus': 'German functional modernism (1919-33): plain cubic forms, flat roofs, white walls, large windows.',
 'Beaux-Arts': 'Late 19th century classical grandeur: symmetrical facades, columns, sculpture, mansard roofs.',
 'Brutalist': 'Mid-20th century: raw exposed concrete in massive, blocky forms.',
 'Buddhist': 'Temples, stupas, pagodas and monasteries with symmetrical layouts and Buddha imagery.',
 'Byzantine': 'Eastern Roman: large domes on round arches, rich mosaics, as in Hagia Sophia.',
 'Chicago school': 'Late 19th century early skyscrapers: steel frames, large windows, little ornament.',
 'Colonial': 'Settler-era style: symmetrical fronts, simple classical details, columned porches.',
 'Deconstructivism': 'Late 20th century: fragmented, angular, distorted shapes that break the regular box.',
 'Dravidian': 'South Indian temples: towering carved gopuram gateways, pillared halls and temple tanks.',
 'Edwardian': 'Early 1900s British: lighter than Victorian, red brick, big windows, restrained ornament.',
 'Georgian': 'British (1714-1830): symmetrical brick fronts, sash windows, classical proportions.',
 'Gothic': 'Medieval European: pointed arches, ribbed vaults, flying buttresses and tall spires.',
 'Greek Revival': 'Early-mid 19th century: temple-like fronts with columns and triangular pediments.',
 'Hausa': 'West African mud-brick buildings with decorative relief patterns.',
 'International style': 'Mid-20th century: glass-and-steel boxes, flat roofs, no ornament.',
 'Khmer': 'Cambodian (Angkor): sandstone temple mountains, towers, galleries and carved reliefs.',
 'Maya': 'Mesoamerican: stepped stone pyramids, temple platforms and corbelled vaults.',
 'Minangkabau': 'West Sumatra: rumah gadang houses with sweeping upturned horn-shaped roofs.',
 'Moorish': 'Islamic Iberia and North Africa: horseshoe arches, geometric tilework, courtyards.',
 'Mughal': 'Indo-Islamic (16th-18th century): onion domes, white marble, minarets, symmetrical layout, as in the Taj Mahal.',
 'Novelty': 'Buildings shaped like objects or with playful, eye-catching roadside designs.',
 'Ottoman': 'Large central domes with semi-domes and slender pencil-shaped minarets.',
 'Palladian': 'Inspired by Palladio: strict symmetry, temple-front porticos, classical villas.',
 'Postmodern': 'Late 20th century: playful mix of historical references, bold colours and ornament.',
 'Queen Anne': 'Late Victorian houses: asymmetry, turrets, wraparound porches, decorative shingles.',
 'Romanesque': '11th-12th century: thick walls, round arches, small windows, sturdy towers.',
 'Russian Revival': '19th century: onion domes, bright colours and ornamental patterns from medieval Russian churches.',
 'Safavid': 'Persian (1501-1736): tiled domes, iwans, blue and turquoise mosaics, as in Isfahan.',
 'Stalinist': 'Soviet mid-century: monumental symmetrical buildings and tiered "wedding cake" towers.',
 'Swahili': 'East African coast: coral-stone buildings, carved wooden doors, Islamic influence.',
 'Tudor Revival': 'Half-timbering, steep gables, tall chimneys and decorative brickwork.',
}
print('Styles without a description:', [s for s in short if s not in DESC])   # should print []

def run(img):
    if img is None:
        return {}, 'Please upload a photo.', ''
    score, closest = building_score(img)
    if score < BUILDING_MIN:
        return {}, (f'This does not look like a building (closest match: "{closest}"). '
                    f'Please upload a photo of a building.'), ''
    res = predict_style(img, k=3)
    name, p = res[0]
    if p >= 0.60:
        msg = f'Looks like {name} architecture ({p:.0%} confident).'
    elif p >= 0.35:
        msg = f'Probably {name}, but not certain ({p:.0%}). Check the other guesses.'
    else:
        msg = (f'Not sure. Best guess is {name} ({p:.0%}). The photo may be old or damaged, '
               f'or the style may not be one of the 43 the model knows.')
    return {n: pr for n, pr in res}, msg, DESC.get(name, '')

demo = gr.Interface(
    fn=run,
    inputs=gr.Image(type='pil', label='Upload a photo of a building'),
    outputs=[gr.Label(num_top_classes=3, label='Top 3 styles'),
             gr.Textbox(label='Result'),
             gr.Textbox(label='About the top guess')],
    title='Heritage Lens: Architectural Style Finder',
    description='Upload a photo (old or new) of a building and the model guesses its architectural style.',
)
demo.launch(share=True, debug=True)

In [ ]:
import glob, os
from PIL import Image

print('--- NON-buildings (want LOW scores) ---')
for f in sorted(glob.glob(f'{PROJECT}/negative_tests/*.*')):
    s, c = building_score(Image.open(f))
    print(f'{os.path.basename(f):25s} building score {s:.2f}   closest: {c}')

print('\n--- Buildings (want HIGH scores) ---')
for f in sorted(glob.glob(f'{PROJECT}/test_images/*.*')):
    s, c = building_score(Image.open(f))
    print(f'{os.path.basename(f):25s} building score {s:.2f}')

In [ ]:
BUILDING_MIN = 0.2
print('Cut-off set to', BUILDING_MIN)

In [ ]:
!pip -q install diffusers accelerate opencv-python-headless

import cv2, numpy as np, torch
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

controlnet = ControlNetModel.from_pretrained('lllyasviel/control_v11p_sd15_canny', torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5', controlnet=controlnet,
    torch_dtype=torch.float16, safety_checker=None).to('cuda')
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
print('Restoration model ready')

In [ ]:
import matplotlib.pyplot as plt

def imagine_restored(img, style=None, cond=1.0, seed=0, steps=25):
    img = img.convert('RGB')
    s = 512 / max(img.size)
    w, h = int(img.width * s) // 8 * 8, int(img.height * s) // 8 * 8
    small = img.resize((w, h))
    edges = cv2.Canny(np.array(small), 100, 200)
    edges = Image.fromarray(np.stack([edges] * 3, -1))
    what = f'{style} building' if style else 'building'
    prompt = f'a photo of an intact, well-preserved {what}, restored, clean facade, sharp details, natural daylight'
    neg = 'ruins, rubble, damage, cracks, broken, graffiti, scaffolding, people, text, watermark, blurry, low quality'
    gen = torch.Generator('cuda').manual_seed(seed)
    out = pipe(prompt, image=edges, negative_prompt=neg, num_inference_steps=steps,
               controlnet_conditioning_scale=cond, guidance_scale=7.5, generator=gen).images[0]
    return small, edges, out

def show(path, cond=1.0, seed=0):
    img = Image.open(path)
    name, p = predict_style(img, 3)[0]
    style = name if p >= 0.60 else None
    o, e, r = imagine_restored(img, style, cond, seed)
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    for a, im, t in zip(ax, [o, e, r], ['Original', 'Edges used', f'AI-imagined restoration (style: {style or "none"})']):
        a.imshow(im); a.set_title(t, fontsize=10); a.axis('off')
    plt.tight_layout(); plt.show()

show(f'{PROJECT}/test_images/damaged.jpg')
show(f'{PROJECT}/test_images/oldtemple.jpg')

In [ ]:
def imagine_restored(img, style=None, cond=1.0, seed=0, steps=30,
                     lo=100, hi=200, guidance=6.0, blur=False):
    img = img.convert('RGB')
    s = 512 / max(img.size)
    w, h = int(img.width * s) // 8 * 8, int(img.height * s) // 8 * 8
    small = img.resize((w, h))
    arr = np.array(small)
    if blur: arr = cv2.GaussianBlur(arr, (5, 5), 0)
    edges = cv2.Canny(arr, lo, hi)
    edges = Image.fromarray(np.stack([edges] * 3, -1))
    what = f'{style} building' if style else 'building'
    prompt = (f'a realistic photograph of an intact, well-maintained {what}, restored, clean facade, '
              f'natural colors, soft daylight, high quality')
    neg = ('ruins, rubble, damage, cracks, broken, graffiti, scaffolding, people, text, watermark, '
           'blurry, low quality, oversaturated, cartoon, painting, illustration, plants growing on walls')
    gen = torch.Generator('cuda').manual_seed(seed)
    out = pipe(prompt, image=edges, negative_prompt=neg, num_inference_steps=steps,
               controlnet_conditioning_scale=cond, guidance_scale=guidance, generator=gen).images[0]
    return small, edges, out

def compare(path, conds=(0.5, 0.75, 1.0), **kw):
    img = Image.open(path)
    name, p = predict_style(img, 3)[0]
    style = name if p >= 0.60 else None
    outs = [imagine_restored(img, style, cond=c, **kw) for c in conds]
    fig, ax = plt.subplots(1, len(conds) + 1, figsize=(5 * (len(conds) + 1), 5))
    ax[0].imshow(outs[0][0]); ax[0].set_title('Original')
    for a, (o, e, r), c in zip(ax[1:], outs, conds):
        a.imshow(r); a.set_title(f'cond={c}')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()

compare(f'{PROJECT}/test_images/damaged.jpg', lo=150, hi=250, blur=True)

In [ ]:
import shutil
RT = f'{PROJECT}/restoration_tests'
os.makedirs(RT, exist_ok=True)
for n in ['oldtemple.jpg', 'damaged.jpg']:
    src = f'{PROJECT}/test_images/{n}'
    if os.path.exists(src):
        shutil.copy(src, RT)
print(os.listdir(RT))

In [ ]:
import os, glob, cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.metrics import structural_similarity as ssim

df = compare_variants(f'{PROJECT}/restoration_tests',
                      extras={'damaged.jpg': 'grey concrete apartment block, overcast sky'})
df

In [ ]:
import shutil
RT = f'{PROJECT}/restoration_tests'
os.makedirs(RT, exist_ok=True)
for n in ['gothic.jpg', 'mughal.jpg']:
    shutil.copy(f'{PROJECT}/test_images/{n}', RT)
print(os.listdir(RT))

In [ ]:
df = compare_variants(f'{PROJECT}/restoration_tests',
                      extras={'damaged.jpg': 'grey concrete apartment block, overcast sky'})
df

In [ ]:
!pip -q install kaggle lpips
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ.pop('KAGGLE_API_TOKEN', None)
if not os.path.exists('/content/data'):
    os.system('kaggle datasets download -d jungseolin/international-architectural-styles-combined -p /content/data --unzip')
print(os.listdir('/content/data')[:5])

In [ ]:
import numpy as np, pandas as pd, cv2, torch, lpips
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageDraw
from skimage.metrics import structural_similarity as ssim, peak_signal_noise_ratio as psnr
from tqdm.auto import tqdm

# ---------- synthetic damage ----------
def sepia_old(img, seed):
    r = np.random.default_rng(seed)
    a = np.asarray(img.convert('L'), dtype=np.float32) / 255.0
    a = 0.25 + 0.55 * a                                    # faded, low contrast
    t = np.stack([a * 1.00, a * 0.85, a * 0.65], -1)       # sepia tint
    t += r.normal(0, 0.04, t.shape)                        # film grain
    h, w = a.shape
    yy, xx = np.mgrid[0:h, 0:w]
    vig = 1 - 0.35 * (((xx - w / 2) / (w / 2)) ** 2 + ((yy - h / 2) / (h / 2)) ** 2) / 2
    t *= vig[..., None]
    out = Image.fromarray((np.clip(t, 0, 1) * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(1.2))
    d = ImageDraw.Draw(out)
    for _ in range(6):                                     # scratches
        x = int(r.integers(0, w)); d.line([(x, 0), (x + int(r.integers(-10, 10)), h)], fill=(235, 225, 205), width=1)
    return out

def damage_patches(img, seed, frac=0.2):
    r = np.random.default_rng(seed)
    a = np.array(img).copy(); h, w = a.shape[:2]
    mask = np.zeros((h, w), np.uint8)
    while mask.mean() / 255 < frac:
        cx, cy = int(r.integers(0, w)), int(r.integers(0, h))
        ax, ay = int(r.integers(w // 20, w // 7)), int(r.integers(h // 20, h // 7))
        cv2.ellipse(mask, (cx, cy), (ax, ay), int(r.integers(0, 180)), 0, 360, 255, -1)
    noise = r.integers(40, 120, a.shape, dtype=np.uint8)
    a[mask > 0] = noise[mask > 0]
    return Image.fromarray(a)

DEGS = {'old_photo': sepia_old,
        'damage': damage_patches,
        'both': lambda im, s: damage_patches(sepia_old(im, s), s)}

# ---------- metrics ----------
lp = lpips.LPIPS(net='alex').to(dev)
def to_t(img):
    return (torch.from_numpy(np.array(img.convert('RGB'))).permute(2, 0, 1).float() / 127.5 - 1).unsqueeze(0).to(dev)
def metrics(ref, test):
    a, b = np.array(ref.convert('RGB')), np.array(test.convert('RGB'))
    assert a.shape == b.shape, (a.shape, b.shape)
    with torch.no_grad(): l = lp(to_t(ref), to_t(test)).item()
    return psnr(a, b, data_range=255), ssim(a, b, channel_axis=2, data_range=255), l

clean = lambda s: s.replace(' architecture', '').replace('Architecture of the ', '').replace('Architecture of ', '')

# ---------- pick 12 intact test photos from different styles ----------
sp = pd.read_csv(f'{PROJECT}/datasets/splits.csv')
test = sp[sp.split == 'test']
N = 12
picks = []
for label in np.random.default_rng(0).permutation(test.label.unique()):
    for _, row in test[test.label == label].sample(frac=1, random_state=1).iterrows():
        if min(Image.open(f'/content/data/{row.path}').size) >= 400:
            picks.append(row); break
    if len(picks) >= N: break
print(len(picks), 'photos selected')

# ---------- run ----------
VARS = {'A: edges only':      dict(mode='edges', cond=0.75),
        'B1: strength 0.75':  dict(mode='img2img', strength=0.75, cond=0.8, min_area=20),
        'B2: strength 0.55':  dict(mode='img2img', strength=0.55, cond=0.8, min_area=20)}
rows, examples = [], []
for k, row in enumerate(tqdm(picks)):
    orig = prep(Image.open(f'/content/data/{row.path}').convert('RGB'))
    true = clean(row.label)
    for dname, dfn in DEGS.items():
        deg = dfn(orig, k)
        n0, c0 = predict_style(deg, 1)[0]
        style = n0 if c0 >= 0.60 else None
        entries = {'No restoration (damaged input)': deg}
        for vname, kw in VARS.items():
            entries[vname] = restore_v(deg, style, seed=0, **kw)[2]
        for vname, im in entries.items():
            p, s, l = metrics(orig, im)
            rows.append(dict(image=k, true_style=true, degradation=dname, variant=vname,
                             psnr=p, ssim=s, lpips=l, style_correct=predict_style(im, 1)[0][0] == true))
        if dname == 'both' and k < 4:
            examples.append((orig, entries, true))
    pd.DataFrame(rows).to_csv(f'{PROJECT}/outputs/synthetic_damage_results.csv', index=False)

res = pd.DataFrame(rows)
cols = ['psnr', 'ssim', 'lpips', 'style_correct']
print('\n=== OVERALL (mean over all photos and damage types) ===')
print(res.groupby('variant')[cols].mean().round(3))
print('\n=== BY DAMAGE TYPE ===')
print(res.groupby(['degradation', 'variant'])[cols].mean().round(3))

# ---------- example figure ----------
names = ['Original'] + list(examples[0][1].keys())
fig, ax = plt.subplots(len(examples), len(names), figsize=(4 * len(names), 3.4 * len(examples)))
for r, (orig, ent, true) in enumerate(examples):
    for c, im in enumerate([orig] + list(ent.values())):
        ax[r, c].imshow(im); ax[r, c].axis('off')
        if r == 0: ax[r, c].set_title(names[c], fontsize=9)
    ax[r, 0].set_title(f'{true}' if r else f'Original\n{true}', fontsize=9)
plt.tight_layout()
plt.savefig(f'{PROJECT}/outputs/synthetic_damage_examples.png', dpi=130)
plt.show()

In [ ]:
from diffusers import StableDiffusionInpaintPipeline
inp = StableDiffusionInpaintPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-inpainting',
    torch_dtype=torch.float16, safety_checker=None).to('cuda')

def make_mask(h, w, seed, frac=0.2):
    r = np.random.default_rng(seed)
    m = np.zeros((h, w), np.uint8)
    while m.mean() / 255 < frac:
        cv2.ellipse(m, (int(r.integers(0, w)), int(r.integers(0, h))),
                    (int(r.integers(w // 20, w // 7)), int(r.integers(h // 20, h // 7))),
                    int(r.integers(0, 180)), 0, 360, 255, -1)
    return m

def fill_damage(img, mask, seed):
    a = np.array(img).copy()
    a[mask > 0] = np.random.default_rng(seed).integers(40, 120, a.shape, dtype=np.uint8)[mask > 0]
    return Image.fromarray(a)

def cv_inpaint(deg, mask):
    return Image.fromarray(cv2.inpaint(np.array(deg), mask, 5, cv2.INPAINT_TELEA))

def sd_inpaint(deg, mask, style, seed=0):
    what = f'{style} building' if style else 'building'
    prompt = f'a realistic photograph of a {what}, intact, natural colors, consistent with the surroundings'
    out = inp(prompt=prompt, image=deg, mask_image=Image.fromarray(mask), negative_prompt=NEG,
              num_inference_steps=30, guidance_scale=7.0, height=deg.height, width=deg.width,
              generator=torch.Generator('cuda').manual_seed(seed)).images[0].resize(deg.size)
    m = (mask > 0)[..., None]                      # keep real pixels outside the mask
    return Image.fromarray(np.where(m, np.array(out), np.array(deg)).astype(np.uint8))

def denoise(img):
    a = cv2.fastNlMeansDenoisingColored(np.array(img), None, 8, 8, 7, 21)
    lab = cv2.cvtColor(a, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

rows, ex_mask, ex_old = [], [], []
def log(k, orig, true, dname, ent):
    for vname, im in ent.items():
        if im.size != orig.size: im = im.resize(orig.size)
        p, s, l = metrics(orig, im)
        rows.append(dict(image=k, true_style=true, degradation=dname, variant=vname,
                         psnr=p, ssim=s, lpips=l, style_correct=predict_style(im, 1)[0][0] == true))

for k, row in enumerate(tqdm(picks)):
    orig = prep(Image.open(f'/content/data/{row.path}').convert('RGB'))
    true = clean(row.label)

    # 1) missing regions, mask known
    mask = make_mask(orig.height, orig.width, k)
    deg = fill_damage(orig, mask, k)
    n0, c0 = predict_style(deg, 1)[0]
    ent = {'No restoration': deg, 'OpenCV inpaint': cv_inpaint(deg, mask),
           'SD inpaint (mask)': sd_inpaint(deg, mask, n0 if c0 >= 0.6 else None)}
    log(k, orig, true, 'missing regions', ent)
    if k < 4: ex_mask.append((orig, ent, true))

    # 2) old photo: clean up first, then gentle redraw
    deg2 = sepia_old(orig, k)
    den = denoise(deg2)
    n1, c1 = predict_style(den, 1)[0]
    ent2 = {'No restoration': deg2, 'Denoise + contrast': den,
            'Denoise + gentle img2img (0.35)': restore_v(den, n1 if c1 >= 0.6 else None, mode='img2img',
                                                         strength=0.35, cond=1.0, min_area=40)[2]}
    log(k, orig, true, 'old photo', ent2)
    if k < 4: ex_old.append((orig, ent2, true))
    pd.DataFrame(rows).to_csv(f'{PROJECT}/outputs/synthetic_damage_results_v2.csv', index=False)

res2 = pd.DataFrame(rows)
print(res2.groupby(['degradation', 'variant'])[['psnr', 'ssim', 'lpips', 'style_correct']].mean().round(3))

def show(examples, fname):
    names = ['Original'] + list(examples[0][1].keys())
    fig, ax = plt.subplots(len(examples), len(names), figsize=(4 * len(names), 3.4 * len(examples)))
    for r, (orig, ent, true) in enumerate(examples):
        for c, im in enumerate([orig] + list(ent.values())):
            ax[r, c].imshow(im); ax[r, c].axis('off')
            if r == 0: ax[r, c].set_title(names[c], fontsize=9)
    plt.tight_layout(); plt.savefig(f'{PROJECT}/outputs/{fname}', dpi=130); plt.show()

show(ex_mask, 'v2_missing_regions.png')
show(ex_old, 'v2_old_photo.png')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install diffusers accelerate transformers gradio opencv-python-headless

import os, json, cv2, numpy as np, torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from transformers import CLIPModel, CLIPProcessor
from diffusers import StableDiffusionInpaintPipeline

PROJECT = '/content/drive/MyDrive/heritage-lens'
dev = 'cuda'

# ---- style classifier ----
classes = json.load(open(f'{PROJECT}/models/classes.json'))
short = [c.replace(' architecture', '').replace('Architecture of the ', '').replace('Architecture of ', '') for c in classes]
clf = models.resnet50()
clf.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(clf.fc.in_features, len(classes)))
clf.load_state_dict(torch.load(f'{PROJECT}/models/best.pt', map_location=dev, weights_only=False)['model'])
clf.to(dev).eval()
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def predict_style(img, k=3):
    img = img.convert('RGB')
    with torch.no_grad():
        p = torch.softmax(clf(tf(img).unsqueeze(0).to(dev)), 1)[0].cpu()
    top = p.topk(k)
    return [(short[i], float(v)) for v, i in zip(top.values, top.indices)]

# ---- "is this a building?" check ----
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(dev).eval()
clip_proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
LABELS = ['a photo of a building', 'a photo of a historic monument or temple',
          'a photo of a ruined or damaged building', 'a photo of a person',
          'a selfie or portrait of a face', 'a photo of an animal', 'a photo of food',
          'a photo of a landscape or nature', 'a photo of a vehicle',
          'a photo of an indoor room', 'a screenshot or a document', 'a photo of an everyday object']
def building_score(img):
    inputs = clip_proc(text=LABELS, images=img.convert('RGB'), return_tensors='pt', padding=True).to(dev)
    with torch.no_grad():
        p = clip(**inputs).logits_per_image.softmax(1)[0].cpu()
    return float(p[:3].sum()), LABELS[int(p.argmax())]

# ---- helpers and the inpainting model ----
def prep(img):
    img = img.convert('RGB')
    s = 512 / max(img.size)
    return img.resize((int(img.width * s) // 8 * 8, int(img.height * s) // 8 * 8))

NEG = ('ruins, rubble, damage, cracks, broken, graffiti, scaffolding, people, text, watermark, '
       'blurry, low quality, oversaturated, cartoon, painting, illustration, plants growing on walls')

inp = StableDiffusionInpaintPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-inpainting',
    torch_dtype=torch.float16, safety_checker=None).to('cuda')

def sd_inpaint(deg, mask, style, seed=0):
    what = f'{style} building' if style else 'building'
    prompt = f'a realistic photograph of a {what}, intact, natural colors, consistent with the surroundings'
    out = inp(prompt=prompt, image=deg, mask_image=Image.fromarray(mask), negative_prompt=NEG,
              num_inference_steps=30, guidance_scale=7.0, height=deg.height, width=deg.width,
              generator=torch.Generator('cuda').manual_seed(seed)).images[0].resize(deg.size)
    m = (mask > 0)[..., None]                      # keep the real pixels outside the mask
    return Image.fromarray(np.where(m, np.array(out), np.array(deg)).astype(np.uint8))

def denoise(img):
    a = cv2.fastNlMeansDenoisingColored(np.array(img), None, 8, 8, 7, 21)
    lab = cv2.cvtColor(a, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

print('Ready')

In [ ]:
import requests, re, time, io, pandas as pd

UA = {'User-Agent': 'HeritageLensStudentProject/1.0 (https://github.com/vigneshhh34/Heritage-Lens)'}
OUT = f'{PROJECT}/wiki_images'
os.makedirs(OUT, exist_ok=True)
QUERIES = ['Brihadisvara Temple Thanjavur', 'Meenakshi Temple Madurai', 'Taj Mahal', 'Red Fort Delhi',
           'Charminar Hyderabad', 'Mysore Palace', 'Notre-Dame de Paris facade', 'Cologne Cathedral',
           'Hagia Sophia', 'Blue Mosque Istanbul', 'Angkor Wat', 'Borobudur', 'Parthenon Athens',
           'Colosseum Rome', 'Sagrada Familia', 'Chrysler Building', 'Victoria Memorial Kolkata',
           'Gateway of India Mumbai', 'Forbidden City Beijing', 'Alhambra Granada']

def strip(h): return re.sub('<[^>]+>', '', h or '').strip()
rows = []
for q in QUERIES:
    try:
        r = requests.get('https://commons.wikimedia.org/w/api.php', headers=UA, timeout=30, params=dict(
            action='query', format='json', generator='search', gsrsearch=f'{q} filetype:bitmap',
            gsrnamespace=6, gsrlimit=15, prop='imageinfo', iiprop='url|mime|size|extmetadata', iiurlwidth=1280))
        pages = (r.json().get('query') or {}).get('pages', {})
    except Exception as e:
        print('search failed:', q, e); continue
    got = 0
    for p in sorted(pages.values(), key=lambda x: x.get('index', 99)):
        ii = (p.get('imageinfo') or [{}])[0]
        if ii.get('mime') != 'image/jpeg' or ii.get('width', 0) < 1000 or ii.get('height', 0) < 700:
            continue
        md = ii.get('extmetadata', {})
        lic = md.get('LicenseShortName', {}).get('value', '')
        if not (lic.startswith('CC') or 'ublic domain' in lic):
            continue
        try:
            img = Image.open(io.BytesIO(requests.get(ii.get('thumburl') or ii['url'], headers=UA, timeout=60).content)).convert('RGB')
        except Exception:
            continue
        if building_score(img)[0] < 0.9:
            continue
        name = re.sub(r'\W+', '_', q)[:28] + f'_{got}.jpg'
        img.save(f'{OUT}/{name}', quality=95)
        rows.append(dict(file=name, query=q, title=p['title'], artist=strip(md.get('Artist', {}).get('value')),
                         license=lic, page=ii.get('descriptionurl')))
        got += 1
        time.sleep(1)
        if got >= 2: break
    time.sleep(1)
pd.DataFrame(rows).to_csv(f'{OUT}/attribution.csv', index=False)
print(len(rows), 'images saved to', OUT)

In [ ]:
!pip -q install lpips
import glob, numpy as np, pandas as pd, cv2, torch, lpips
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from skimage.metrics import structural_similarity as ssim, peak_signal_noise_ratio as psnr

lp = lpips.LPIPS(net='alex').to(dev)
def to_t(img):
    return (torch.from_numpy(np.array(img.convert('RGB'))).permute(2, 0, 1).float() / 127.5 - 1).unsqueeze(0).to(dev)
def metrics(ref, test, md):
    a, b = np.array(ref.convert('RGB')), np.array(test.convert('RGB'))
    with torch.no_grad(): l = lp(to_t(ref), to_t(test)).item()
    mae = np.abs(a.astype(float) - b.astype(float))[md > 0].mean()
    return psnr(a, b, data_range=255), ssim(a, b, channel_axis=2, data_range=255), l, mae

# ---------- damage masks ----------
def m_blobs(h, w, seed, frac=0.2):
    r = np.random.default_rng(seed); m = np.zeros((h, w), np.uint8)
    while m.mean() / 255 < frac:
        cv2.ellipse(m, (int(r.integers(0, w)), int(r.integers(0, h))),
                    (int(r.integers(w // 20, w // 7)), int(r.integers(h // 20, h // 7))),
                    int(r.integers(0, 180)), 0, 360, 255, -1)
    return m
def m_cracks(h, w, seed):
    r = np.random.default_rng(seed); m = np.zeros((h, w), np.uint8)
    for _ in range(int(r.integers(3, 6))):
        x, y = int(r.integers(0, w)), int(r.integers(0, h // 2)); pts = [(x, y)]
        for _ in range(int(r.integers(12, 25))):
            x = int(np.clip(x + r.integers(-25, 26), 0, w - 1)); y = int(np.clip(y + r.integers(5, 30), 0, h - 1)); pts.append((x, y))
        cv2.polylines(m, [np.array(pts, np.int32)], False, 255, int(r.integers(3, 7)))
    return m
def m_patch(h, w, seed, frac=0.25):
    r = np.random.default_rng(seed); m = np.zeros((h, w), np.uint8)
    base = np.sqrt(frac * h * w / np.pi)
    cv2.ellipse(m, (int(r.integers(w // 4, 3 * w // 4)), int(r.integers(h // 4, 3 * h // 4))),
                (int(base * 1.2), int(base / 1.2)), int(r.integers(0, 180)), 0, 360, 255, -1)
    return m
MASKS = {'blobs (20%)': m_blobs, 'cracks': m_cracks, 'one large patch': m_patch}

def fill_noise(img, md, seed):
    a = np.array(img).copy()
    a[md > 0] = np.random.default_rng(seed).integers(40, 120, a.shape, dtype=np.uint8)[md > 0]
    return Image.fromarray(a)

# ---------- generation and blending ----------
def sd_raw(deg, md, style, seed):
    what = f'{style} building' if style else 'building'
    prompt = f'a realistic photograph of a {what}, intact, natural colors, consistent with the surroundings'
    out = inp(prompt=prompt, image=deg, mask_image=Image.fromarray(md), negative_prompt=NEG,
              num_inference_steps=30, guidance_scale=7.0, height=deg.height, width=deg.width,
              generator=torch.Generator('cuda').manual_seed(seed)).images[0]
    return out.resize(deg.size)

def hard(deg, gen, md):
    return Image.fromarray(np.where((md > 0)[..., None], np.array(gen), np.array(deg)).astype(np.uint8))

def soft(deg, gen, md):
    d, g = np.array(deg).astype(np.float32), np.array(gen).astype(np.float32)
    m = md > 0
    ring = (cv2.dilate(md, np.ones((41, 41), np.uint8)) > 0) & ~m
    if ring.sum() > 100 and m.sum() > 100:
        g = np.clip(g + 0.5 * (d[ring].mean(0) - g[m].mean(0)), 0, 255)   # partial colour match
    a = cv2.GaussianBlur(cv2.dilate(md, np.ones((7, 7), np.uint8)).astype(np.float32) / 255, (0, 0), 3)
    a = np.maximum(a, m.astype(np.float32))[..., None]
    return Image.fromarray((a * g + (1 - a) * d).astype(np.uint8))

def seam_score(deg, cand, md):          # lower = fill matches its surroundings better
    c = cv2.cvtColor(np.array(cand), cv2.COLOR_RGB2LAB).astype(np.float32)
    d = cv2.cvtColor(np.array(deg), cv2.COLOR_RGB2LAB).astype(np.float32)
    m = md > 0
    ring = (cv2.dilate(md, np.ones((41, 41), np.uint8)) > 0) & ~m
    if ring.sum() < 100 or m.sum() < 100: return 0.0
    ci, dr = c[m], d[ring]
    return float(np.abs(ci.mean(0) - dr.mean(0)).mean() + 0.5 * np.abs(ci.std(0) - dr.std(0)).mean())

# ---------- run ----------
files = sorted(glob.glob(f'{PROJECT}/wiki_images/*.jpg'))
files = files[::max(1, len(files) // 12)][:12]
print(len(files), 'photos')
rows, examples = [], []
for k, f in enumerate(tqdm(files)):
    orig = prep(Image.open(f).convert('RGB'))
    for mname, mfn in MASKS.items():
        mask = mfn(orig.height, orig.width, k)
        md = cv2.dilate(mask, np.ones((9, 9), np.uint8))
        deg = fill_noise(orig, md, k)
        n0, c0 = predict_style(deg, 1)[0]
        style = n0 if c0 >= 0.6 else None
        raws = [sd_raw(deg, md, style, s) for s in range(4)]
        softs = [soft(deg, r, md) for r in raws]
        pick = int(np.argmin([seam_score(deg, s, md) for s in softs]))
        lps = []
        for s in softs:
            with torch.no_grad(): lps.append(lp(to_t(orig), to_t(s)).item())
        ent = {'No restoration': deg,
               'A: current (hard paste)': hard(deg, raws[0], md),
               'B: soft blend + colour match': softs[0],
               'C: best of 4 (auto pick)': softs[pick],
               'Oracle best of 4 (upper bound)': softs[int(np.argmin(lps))]}
        for vname, im in ent.items():
            p, s_, l, mae = metrics(orig, im, md)
            rows.append(dict(photo=k, mask=mname, variant=vname, psnr=p, ssim=s_, lpips=l, mae_in_mask=mae,
                             seed_lpips_std=float(np.std(lps))))
        if k < 4: examples.append((mname, orig, ent))
    pd.DataFrame(rows).to_csv(f'{PROJECT}/outputs/inpaint_round2.csv', index=False)

res = pd.DataFrame(rows)
order = list(ent.keys())
print('\n=== OVERALL ===')
print(res.groupby('variant')[['psnr', 'ssim', 'lpips', 'mae_in_mask']].mean().loc[order].round(3))
print('\n=== BY DAMAGE TYPE ===')
print(res.groupby(['mask', 'variant'])[['psnr', 'ssim', 'lpips', 'mae_in_mask']].mean().round(3))
print('\nMean spread of LPIPS between the 4 seeds:', round(res.seed_lpips_std.mean(), 3))

# example picture: first 4 photos, large patch
ex = [e for e in examples if e[0] == 'one large patch'][:4]
names = ['Original'] + list(ex[0][2].keys())
fig, ax = plt.subplots(len(ex), len(names), figsize=(3.6 * len(names), 3.2 * len(ex)))
ax = np.atleast_2d(ax)
for r_, (_, orig, ent_) in enumerate(ex):
    for c_, im in enumerate([orig] + list(ent_.values())):
        ax[r_, c_].imshow(im); ax[r_, c_].axis('off')
        if r_ == 0: ax[r_, c_].set_title(names[c_], fontsize=8)
plt.tight_layout(); plt.savefig(f'{PROJECT}/outputs/inpaint_round2_examples.png', dpi=130); plt.show()

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
blip_proc = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
blip = BlipForConditionalGeneration.from_pretrained('Salesforce/blip-image-captioning-base',
                                                    torch_dtype=torch.float16).to(dev)

def caption(img):
    x = blip_proc(images=img, text='a photograph of', return_tensors='pt').to(dev, torch.float16)
    with torch.no_grad():
        out = blip.generate(**x, max_new_tokens=30)
    return blip_proc.decode(out[0], skip_special_tokens=True).strip()

def clip_emb(imgs):
    x = clip_proc(text=['a photo'], images=imgs, return_tensors='pt', padding=True).to(dev)
    with torch.no_grad():
        return torch.nn.functional.normalize(clip(**x).image_embeds, dim=-1)
def clip_sim(a, b):
    e = clip_emb([a, b]); return float((e[0] * e[1]).sum())
def clip_text(img, text):
    x = clip_proc(text=[text], images=img, return_tensors='pt', padding=True, truncation=True).to(dev)
    with torch.no_grad():
        return float(clip(**x).logits_per_image[0, 0])
def bbox(md, pad=16):
    ys, xs = np.where(md > 0)
    return (max(xs.min() - pad, 0), max(ys.min() - pad, 0), min(xs.max() + pad, md.shape[1]), min(ys.max() + pad, md.shape[0]))

def sd_prompt(deg, md, prompt, seed):
    out = inp(prompt=prompt, image=deg, mask_image=Image.fromarray(md), negative_prompt=NEG,
              num_inference_steps=30, guidance_scale=7.0, height=deg.height, width=deg.width,
              generator=torch.Generator('cuda').manual_seed(seed)).images[0]
    return out.resize(deg.size)

rows, caps, examples = [], [], []
for k, f in enumerate(tqdm(files)):
    orig = prep(Image.open(f).convert('RGB'))
    for mname, mfn in MASKS.items():
        mask = mfn(orig.height, orig.width, k)
        md = cv2.dilate(mask, np.ones((9, 9), np.uint8))
        deg = fill_noise(orig, md, k)
        n0, c0 = predict_style(deg, 1)[0]
        style = n0 if c0 >= 0.6 else None
        cap = caption(Image.fromarray(cv2.inpaint(np.array(deg), md, 5, cv2.INPAINT_TELEA)))
        prompt = f'{cap}, intact, restored, undamaged, realistic photograph'
        bb = bbox(md); oc = orig.crop(bb)

        A = hard(deg, sd_raw(deg, md, style, 0), md)                       # current app
        hards = [hard(deg, sd_prompt(deg, md, prompt, s), md) for s in range(4)]
        ts = [clip_text(h.crop(bb), prompt) for h in hards]
        lps, sims = [], []
        for h in hards:
            with torch.no_grad(): lps.append(lp(to_t(orig), to_t(h)).item())
            sims.append(clip_sim(oc, h.crop(bb)))
        ent = {'No restoration': deg,
               'A: generic prompt (current app)': A,
               'P: caption prompt (1 try)': hards[0],
               'P+S: caption + CLIP pick of 4': hards[int(np.argmax(ts))],
               'Oracle LPIPS (best of 4)': hards[int(np.argmin(lps))],
               'Oracle semantic (best of 4)': hards[int(np.argmax(sims))]}
        for vname, im in ent.items():
            p_, s_, l_, mae = metrics(orig, im, md)
            rows.append(dict(photo=k, mask=mname, variant=vname, psnr=p_, ssim=s_, lpips=l_,
                             mae_in_mask=mae, clip_sim_hole=clip_sim(oc, im.crop(bb))))
        caps.append(dict(photo=k, mask=mname, caption=cap))
        if k < 4: examples.append((mname, orig, ent))
    pd.DataFrame(rows).to_csv(f'{PROJECT}/outputs/inpaint_round3.csv', index=False)

res3 = pd.DataFrame(rows); order = list(ent.keys())
cols = ['psnr', 'ssim', 'lpips', 'mae_in_mask', 'clip_sim_hole']
print('=== OVERALL ===');        print(res3.groupby('variant')[cols].mean().loc[order].round(3))
print('\n=== BY DAMAGE TYPE ==='); print(res3.groupby(['mask', 'variant'])[cols].mean().round(3))
print('\nCaptions used:');       print(pd.DataFrame(caps).drop_duplicates('photo').to_string(index=False))

ex = [e for e in examples if e[0] == 'one large patch'][:4]
names = ['Original'] + list(ex[0][2].keys())
fig, ax = plt.subplots(len(ex), len(names), figsize=(3.4 * len(names), 3.0 * len(ex)))
ax = np.atleast_2d(ax)
for r_, (_, o_, e_) in enumerate(ex):
    for c_, im in enumerate([o_] + list(e_.values())):
        ax[r_, c_].imshow(im); ax[r_, c_].axis('off')
        if r_ == 0: ax[r_, c_].set_title(names[c_], fontsize=7)
plt.tight_layout(); plt.savefig(f'{PROJECT}/outputs/inpaint_round3_examples.png', dpi=130); plt.show()

In [ ]:
import gradio as gr, traceback

DESC = {
 'Achaemenid': 'Ancient Persian (c. 550-330 BC): monumental stone columns, grand staircases and relief carvings, as at Persepolis.',
 'Amazigh': 'Berber architecture of North Africa: fortified earthen villages, kasbahs and mud-brick towers.',
 'American Foursquare': 'Boxy American house (c. 1895-1930): hipped roof, wide front porch, central dormer.',
 'American craftsman style': 'Early 20th-century American homes: low gabled roofs, wide eaves, exposed rafters, tapered porch columns.',
 'Ancient Egyptian': 'Massive stone temples and pyramids with huge columns, pylon gateways and hieroglyph carvings.',
 'Edo Period': 'Japanese (1603-1868): wooden buildings, tiled roofs, castles and tea houses.',
 'Joseon Dynasty': 'Korean (1392-1897): curved tiled roofs, timber frames, palaces and hanok houses.',
 'Qing Dynasty': 'Chinese (1644-1912): imperial palaces with glazed-tile roofs, red walls and ornate wooden brackets.',
 'Art Deco': '1920s-30s: geometric, streamlined forms, stepped skyscraper tops and bold decoration.',
 'Art Nouveau': 'c. 1890-1910: flowing organic curves, floral ironwork and decorative glass.',
 'Balinese': 'Indonesian temples and compounds: carved stone gateways, tiered thatched shrines, open pavilions.',
 'Baroque': '17th-18th century European: dramatic curves, rich ornament, grand domes and facades.',
 'Bauhaus': 'German functional modernism (1919-33): plain cubic forms, flat roofs, white walls, large windows.',
 'Beaux-Arts': 'Late 19th century classical grandeur: symmetrical facades, columns, sculpture, mansard roofs.',
 'Brutalist': 'Mid-20th century: raw exposed concrete in massive, blocky forms.',
 'Buddhist': 'Temples, stupas, pagodas and monasteries with symmetrical layouts and Buddha imagery.',
 'Byzantine': 'Eastern Roman: large domes on round arches, rich mosaics, as in Hagia Sophia.',
 'Chicago school': 'Late 19th century early skyscrapers: steel frames, large windows, little ornament.',
 'Colonial': 'Settler-era style: symmetrical fronts, simple classical details, columned porches.',
 'Deconstructivism': 'Late 20th century: fragmented, angular, distorted shapes that break the regular box.',
 'Dravidian': 'South Indian temples: towering carved gopuram gateways, pillared halls and temple tanks.',
 'Edwardian': 'Early 1900s British: lighter than Victorian, red brick, big windows, restrained ornament.',
 'Georgian': 'British (1714-1830): symmetrical brick fronts, sash windows, classical proportions.',
 'Gothic': 'Medieval European: pointed arches, ribbed vaults, flying buttresses and tall spires.',
 'Greek Revival': 'Early-mid 19th century: temple-like fronts with columns and triangular pediments.',
 'Hausa': 'West African mud-brick buildings with decorative relief patterns.',
 'International style': 'Mid-20th century: glass-and-steel boxes, flat roofs, no ornament.',
 'Khmer': 'Cambodian (Angkor): sandstone temple mountains, towers, galleries and carved reliefs.',
 'Maya': 'Mesoamerican: stepped stone pyramids, temple platforms and corbelled vaults.',
 'Minangkabau': 'West Sumatra: rumah gadang houses with sweeping upturned horn-shaped roofs.',
 'Moorish': 'Islamic Iberia and North Africa: horseshoe arches, geometric tilework, courtyards.',
 'Mughal': 'Indo-Islamic (16th-18th century): onion domes, white marble, minarets, symmetrical layout, as in the Taj Mahal.',
 'Novelty': 'Buildings shaped like objects or with playful, eye-catching roadside designs.',
 'Ottoman': 'Large central domes with semi-domes and slender pencil-shaped minarets.',
 'Palladian': 'Inspired by Palladio: strict symmetry, temple-front porticos, classical villas.',
 'Postmodern': 'Late 20th century: playful mix of historical references, bold colours and ornament.',
 'Queen Anne': 'Late Victorian houses: asymmetry, turrets, wraparound porches, decorative shingles.',
 'Romanesque': '11th-12th century: thick walls, round arches, small windows, sturdy towers.',
 'Russian Revival': '19th century: onion domes, bright colours and ornamental patterns from medieval Russian churches.',
 'Safavid': 'Persian (1501-1736): tiled domes, iwans, blue and turquoise mosaics, as in Isfahan.',
 'Stalinist': 'Soviet mid-century: monumental symmetrical buildings and tiered "wedding cake" towers.',
 'Swahili': 'East African coast: coral-stone buildings, carved wooden doors, Islamic influence.',
 'Tudor Revival': 'Half-timbering, steep gables, tall chimneys and decorative brickwork.',
}

def get_mask(editor):
    layers = editor.get('layers') or []
    if not layers: return None
    a = np.array(layers[0].convert('RGBA'))[..., 3]
    m = (a > 10).astype(np.uint8) * 255
    return m if m.any() else None

def run_app(editor, mode):
    if editor is None or editor.get('background') is None:
        return None, {}, 'Please upload a photo.', ''
    img = prep(editor['background'].convert('RGB'))
    score, closest = building_score(img)
    if score < 0.2:
        return None, {}, f'This does not look like a building (closest match: "{closest}").', ''

    res = predict_style(img, 3); name, p = res[0]
    top3 = {n: pr for n, pr in res}
    if p >= 0.60:   msg = f'Style: {name} ({p:.0%} confident).'
    elif p >= 0.35: msg = f'Style: probably {name} ({p:.0%}), not certain.'
    else:           msg = f'Style: not sure, best guess {name} ({p:.0%}).'
    desc = DESC.get(name, '')

    out = img
    if mode.startswith('Fill'):
        mask = get_mask(editor)
        if mask is None:
            return img, top3, msg + '\nPaint over the damaged areas first, then submit again.', desc
        mask = cv2.resize(mask, img.size, interpolation=cv2.INTER_NEAREST)
        frac = (mask > 0).mean()
        if frac > 0.35:
            return img, top3, msg + f'\nYou painted {frac:.0%} of the image. Paint only the damaged areas (under about 35%), otherwise the AI replaces the building instead of repairing it.', desc
        mask = cv2.dilate(mask, np.ones((9, 9), np.uint8))
        out = sd_inpaint(img, mask, name if p >= 0.60 else None)
        msg += '\nDamaged areas filled by AI. This is an illustration, not the true original.'
    elif mode.startswith('Clean'):
        out = denoise(img)
        msg += '\nBasic denoise and contrast. In our tests the effect on old photos was small.'
    return out, top3, msg, desc

def run_safe(editor, mode):
    try:
        return run_app(editor, mode)
    except Exception:
        return None, {}, 'ERROR: ' + traceback.format_exc()[-1500:], ''

try: demo.close()
except Exception: pass

demo = gr.Interface(
    fn=run_safe,
    inputs=[gr.ImageEditor(type='pil', label='Upload a photo, then paint over damaged areas (optional)',
                           brush=gr.Brush(colors=['#ff0000'], color_mode='fixed', default_size=12)),
            gr.Radio(['Style only', 'Fill damaged areas (paint over them)', 'Clean up old photo'],
                     value='Style only', label='What to do')],
    outputs=[gr.Image(label='Result (AI-generated parts are illustrations)'),
             gr.Label(num_top_classes=3, label='Top 3 styles'),
             gr.Textbox(label='Result'),
             gr.Textbox(label='About the top style guess')],
    title='Heritage Lens: Style Finder and Photo Repair',
)
demo.launch(share=True, debug=True)